In [ ]:
!pip install camel_tools

In [ ]:
from camel_tools.ner import NERecognizer
from camel_tools.tokenizers.word import simple_word_tokenize

import re


ModuleNotFoundError: ignored

In [ ]:
import re

def has_no_arabic_chars(word):

    pattern =  r'[«#>\]_*,;%@+/:^)~»<?$؛({\'!}=.\[\\"`،|\-&]'
    return re.sub(pattern, '', word)


def clean_data(word, label):
  if has_no_arabic_chars(word) =='': return [], []
  else : return has_no_arabic_chars(word), label

def split_text_file(filename):
    with open(filename ,'r', encoding='utf-8') as file:
        lines = file.readlines()
    data = [];
    for line in lines:
        line = line.strip()
        if line:
            parts = line.split('\t')
            text = parts[0]
            _sentence = text.split(" ")
            _labels = parts[1:][0].split(" ")
            sentence= []; labels =[]
            for i, word in enumerate(_sentence):
              tmp_sen, tmp_leb = clean_data(word, _labels[i])
              if tmp_sen != []: sentence.append(tmp_sen); labels.append(tmp_leb)
            data.append((sentence, labels))

    return data


data= split_text_file("/content/drive/MyDrive/projects/NER/a.txt")

In [ ]:
ner = NERecognizer('CAMeL-Lab/bert-base-arabic-camelbert-msa-ner')
# sentence = simple_word_tokenize('إمارة أبوظبي هي إحدى إمارات دولة الإمارات العربية المتحدة السبع')
# ner.predict_sentence(sentence)


NameError: ignored

In [ ]:

label_convert = {
    "B-LOC":"loc", "I-LOC":"loc","B-ORG":"org","I-ORG":"org",
    "I-PERS":"per", "B-PERS":"per", "O":"O", "B-COM": "O",
    "I-COM": "O","B-MISC" :"misc", "I-MISC":"misc",
}


def predict_data(sentence, nlp):
  annotations = nlp.predict_sentence(sentence)
  entities = []
  tags = []
  for i, _sentence in enumerate(sentence):
    entities.append(_sentence) ; tags.append(label_convert[annotations[i]])
  return entities, tags

predicted_data = []

data_values = []
count = 0;

for X, y in data:
  if count%50 == 0: print(count); print(X)
  data_values.append((X, y))
  predicted_data.append(predict_data(X, ner))
  count+=1 ;



In [ ]:
len(data_values[0][1]),len( predicted_data[0][1])

(40, 40)

In [ ]:
grouped_data = []
count = 0
for words, true_label in data_values:
    pred_words, pred_label = predicted_data[count]
    count +=1
    for i, word in enumerate(words):
      if word == pred_words[i] :
        grouped_data.append([word, true_label[i], pred_label[i]])
      else:
        grouped_data.append([word, true_label[i], pred_label[i]])

In [ ]:
import pandas as pd



df = pd.DataFrame(grouped_data, columns=['Word', 'Target', 'Predicted'])
df['Target'] = df['Target'].map({
    "B-PERS":"per", "I-PERS"	:"per", "O":"O",
    "B-LOC": "loc",  "B-evnt": "evnt","B-ORG": "org","B-MISC": "misc",
    "I-LOC": "loc",  "I-evnt": "evnt","I-ORG": "org","I-MISC": "misc",
    })

df

,Word,Target,Predicted
0,الصالحية,loc,loc
1,المفرق,loc,loc
2,غيث,per,per
3,الطراونة,per,per
4,أمر,O,O
...,...,...,...
22504,الوزيرة,O,O
22505,ابن,O,O
22506,وابنة,O,O
22507,خالد,per,per


In [ ]:
from sklearn.metrics import classification_report
report = classification_report(list(df['Target']), list(df['Predicted']))
print(report)

              precision    recall  f1-score   support

           O       0.98      0.99      0.99     19139
         loc       0.89      0.93      0.91       751
        misc       0.88      0.52      0.66       398
         org       0.81      0.75      0.78       725
         per       0.94      0.94      0.94      1496

    accuracy                           0.97     22509
   macro avg       0.90      0.83      0.86     22509
weighted avg       0.97      0.97      0.97     22509



In [ ]:
from sklearn.metrics import confusion_matrix
conf_matrix = confusion_matrix(df['Target'], df['Predicted'], labels=['loc', 'misc', 'org', 'per', 'O'])

In [ ]:
conf_matrix

array([[  702,     0,     8,     7,    34],
       [   22,   208,    25,    10,   133],
       [   31,     5,   547,    28,   114],
       [   12,     6,    23,  1407,    48],
       [   20,    17,    71,    50, 18981]])

In [ ]:
digits_convert ={"loc": 1,  "misc": 2,"org": 3, "per": 4, "O": 5,}
df['Target'] = df['Target'].map(digits_convert)
df['Predicted'] = df['Predicted'].map(digits_convert)
df.head()

,Word,Target,Predicted
0,الصالحية,1,1
1,المفرق,1,1
2,غيث,4,4
3,الطراونة,4,4
4,أمر,5,5


In [ ]:
from sklearn.metrics import f1_score
labels_ = {}
f1_score(df['Target'], df['Predicted'], average='macro')

0.855394407345228

In [ ]:
f1_score(df['Target'], df['Predicted'], average='micro')


0.9705006886134435

In [ ]:
f1_score(df['Target'], df['Predicted'], average='weighted')

0.9691423112606243

In [ ]:
f1_scores =f1_score(df['Target'], df['Predicted'], average=None)

In [ ]:

results = pd.DataFrame(conf_matrix, columns=['loc', 'misc', 'org', 'per', 'O'])
results['f1_score'] = f1_scores
results

,loc,misc,org,per,O,f1_score
0,702,0,8,7,34,0.912874
1,22,208,25,10,133,0.656151
2,31,5,547,28,114,0.781987
3,12,6,23,1407,48,0.938626
4,20,17,71,50,18981,0.987334
